# 02. Цифровой ординатор: 3 синтетических кейса

Контур 1 - RAG-план по КР 154_4 с проверкой ссылок · Контур 2 - детерминированная симуляция до 3-х суток и пересмотр плана · Контур 3 - AI-аудитор (правила + LLM, находки LLM перепроверяет вторая модель).

По умолчанию `LLM_MODE=replay`: ответы LLM (планировщик и проверка находок - Qwen3.8-27B на Groq, LLM-аудитор - Nemotron 3 Ultra на OpenRouter) берутся из `data/llm_cache/`, ключи не нужны. Для нового прогона: `LLM_MODE=live`, `GROQ_API_KEY` и `OPENROUTER_API_KEY` в `.env`.

In [1]:
import os, warnings
os.environ['TQDM_DISABLE'] = '1'
os.environ['HF_HUB_DISABLE_PROGRESS_BARS'] = '1'
warnings.filterwarnings('ignore')
import transformers
transformers.utils.logging.disable_progress_bar()
transformers.utils.logging.set_verbosity_error()
from huggingface_hub.utils import logging as hf_logging
hf_logging.set_verbosity_error()  # без предупреждения о запросах к HF Hub без токена

In [2]:
import pandas as pd
from IPython.display import Markdown, display

from resident.auditor.report import finding_line
from resident.llm.context import LAB_NAMES, format_plan
from resident.pipeline import Pipeline, load_case

pipe = Pipeline()  # retriever и режим LLM - из .env
runs = {cid: pipe.run(load_case(cid)) for cid in ['case1', 'case2', 'case3']}
{'planner, replanner, проверка находок': pipe.llm.stats, 'LLM-аудитор': pipe.auditor_llm.stats}

{'planner, replanner, проверка находок': {'cache_hits': 29,
  'api_calls': 0,
  'reasoning_tokens': 0},
 'LLM-аудитор': {'cache_hits': 3, 'api_calls': 0, 'reasoning_tokens': 0}}

In [3]:
def show_plan(run):
    rows = []
    for i in run.plan_final.items:
        rows.append({'id': i.item_id, 'источник': i.source, 'назначение': i.name,
                     'доза/режим': ', '.join(x for x in (i.dose, i.route, i.frequency, i.timing) if x),
                     'ссылка': '; '.join(c.chunk_id.split(':', 1)[1] for c in i.citations),
                     'проверка': i.verification, 'с, ч': i.start_h, 'до, ч': i.stop_h})
    return pd.DataFrame(rows)

def show_labs(run):
    df = pd.DataFrame([(v.t_h, LAB_NAMES[v.code], v.value) for v in run.sim_final.labs], columns=['ч', 'показатель', 'значение'])
    return df.pivot_table(index='показатель', columns='ч', values='значение')

def show_case(run):
    c = run.case
    display(Markdown(f'## {c.case_id}: {c.title}'))
    print(c.patient.anamnesis)
    display(show_plan(run))
    display(show_labs(run))
    for d in run.deviations:
        print(f'[{d.severity}] {d.message}  | порог: {d.source}')
    for ch in run.changes:
        print('\nПЕРЕСМОТР на', ch.t_h, 'ч:', ch.rationale)
        for s in ch.stopped:
            print('  - отменено', s.item_id, '-', s.reason, '|', '; '.join(x.chunk_id for x in s.citations))
        for a in ch.added:
            print('  + добавлено', a.name, a.dose or '', '|', '; '.join(x.chunk_id for x in a.citations), f'({a.verification})')
    print('\nАУДИТ:')
    for f in run.audit.findings:
        print(finding_line(f))
    if run.audit.rejected:
        print('\nОТКЛОНЕНО ПРОВЕРКОЙ ВТОРОЙ МОДЕЛЬЮ:')
        for f in run.audit.rejected:
            print(finding_line(f))
    print('\nЗаложенные дефекты:', run.coverage)

## Кейс 1 - Baseline
ИМбпST, первая линия по КР, ранняя инвазивная стратегия; к 3-м суткам тропонин снижается, почки и Hb стабильны → пересмотр не нужен. Аудит проверяет отсутствие ложных срабатываний.

In [4]:
show_case(runs['case1'])

## case1: Базовый: ИМбпST, типичное течение, первая линия по КР

Давящая боль за грудиной 40 мин в покое, 4 ч назад. Курит 20 лет. АГ 2 ст. На ЭКГ депрессия ST 1,5 мм в V4-V6. Гемодинамика стабильна: ЧСС 64 уд/мин, АД 132/84 мм рт. ст., Killip I. ЭхоКГ: ФВ ЛЖ 55%, гипокинез нижней стенки.


,id,источник,назначение,доза/режим,ссылка,проверка,"с, ч","до, ч"
0,proc-1,external,Коронарография + ЧКВ со стентированием (лучево...,,,not_checked,10.0,NaN
1,lab-1,planner,Креатинин сыворотки крови,,2.3:t04,verified,0.0,NaN
2,lab-2,planner,Общий (клинический) анализ крови,,2.3:t03,verified,0.0,NaN
3,lab-3,planner,Тропонин I (или T) высокочувствительный,,2.3:t01,verified,0.0,NaN
4,lab-4,planner,Тропонин I (или T) высокочувствительный (повто...,,2.3:t02,verified,0.0,NaN
5,lab-5,planner,"Электролиты крови (калий, натрий)",,2.3:t07,verified,0.0,NaN
6,lab-6,planner,"Липидный профиль (ХС, ХС ЛНП, ХС ЛВП, триглице...",,2.3:t06,verified,0.0,NaN
7,lab-7,planner,Глюкоза крови натощак / Гликированный гемоглобин,,2.3:t05,verified,0.0,NaN
8,lab-8,planner,"Коагулограмма (АЧТВ, ПВ, ПИ, фибриноген)",,2.3:t08,verified,0.0,NaN
9,instr-1,planner,Мониторирование электрокардиографических данных,,2.4:t03,verified,0.0,NaN


ч,0.0,1.0,3.0,24.0,48.0,72.0
показатель,,,,,,
Гемоглобин,148.0,NaN,NaN,148.0,146.0,146.0
Глюкоза,5.8,NaN,NaN,5.2,5.7,6.0
Калий,4.3,NaN,NaN,4.5,4.8,4.7
Креатинин,85.0,NaN,NaN,85.0,91.0,90.0
Тромбоциты,232.0,NaN,NaN,235.0,231.0,231.0
Тропонин (вч),120.0,141.0,188.0,231.0,106.0,44.0
ХС ЛНП,3.6,NaN,NaN,NaN,NaN,NaN
рСКФ (CKD-EPI),92.6,NaN,NaN,92.4,85.0,86.4



АУДИТ:

ОТКЛОНЕНО ПРОВЕРКОЙ ВТОРОЙ МОДЕЛЬЮ:
[MINOR] LLM (clinical) apt-1, apt-2 - ИПП не назначен при ДАПТ, хотя по КР 154_4 п. 3.2.3.1 тезис 14 ИПП рекомендуются только при высоком риске ЖКК (язва/ЖКК в анамнезе, хронические НПВС/кортикостероиды, или ≥2 признаков: возраст ≥65 лет, диспепсия, ГЭРБ, H. pylori, алкоголь). У данного пациента высокого риска ЖКК нет (фактор «пожилой возраст ≥75 лет» отсутствует, возраст 56 лет), поэтому назначение ИПП не обязательно. Нарушения КР нет, но аудитор фиксирует обоснование отсутствия ИПП. (факт: Пациент 56 лет, без анамнеза язвенной болезни/ЖКК, без НПВС/кортикостероидов, без H. pylori/алкоголя. Единственный фактор риска ЖКК по КР — возраст ≥65 лет, которого нет. Назначен ДАПТ (apt-1 АСК + apt-2 тикагрелор), ИПП не назначен.)
    основание: 154_4:3.2.3.1:t14: «У пациентов с ОКСбпST и высоким риском желудочно-кишечных кровотечений во время двойной ан…»
    проверка второй моделью: Тезис КР рекомендует ИПП пациентам с ОКСбпST, имеющим высокий риск

## Кейс 2 - Comorbid
ХБП С3б, язва с ЖКК в анамнезе, ангиоотёк на иАПФ, ФВ 38% → модификация стандарта с первого дня (гидратация и ограничение контраста, ИПП, АРА вместо иАПФ; при ФВ 38% и ХСН - бета-адреноблокатор и антагонист альдостерона). В трек заложены внешние назначения приёмного отделения: эналаприл, диклофенак, омепразол.

In [5]:
show_case(runs['case2'])

## case2: Коморбидный: ХБП С3б, язвенный анамнез, ангиоотёк на иАПФ

Боль за грудиной и одышка при ходьбе 6 ч назад, в покое одышки нет. ЧСС 96 уд/мин, АД 148/86 мм рт. ст., Killip I, застоя в лёгких нет. ЭКГ: депрессия ST в I, aVL, V5-V6. ЭхоКГ при поступлении: ФВ ЛЖ 38%, ХСН NYHA II. Язвенная болезнь двенадцатиперстной кишки, желудочно-кишечное кровотечение 2 года назад. ХБП С3б. Ангионевротический отёк на эналаприл в 2019 г.


,id,источник,назначение,доза/режим,ссылка,проверка,"с, ч","до, ч"
0,proc-1,external,"Коронарография ± ЧКВ (лучевой доступ, изоосмол...",,,not_checked,20.0,NaN
1,ext-1,external,Эналаприл,"2,5 мг, внутрь, 2 раза в сутки",,not_checked,2.0,NaN
2,ext-2,external,Диклофенак,"75 мг, в/м, при болях",,not_checked,3.0,NaN
3,ext-3,external,Омепразол,"20 мг, внутрь, 1 раз в сутки",,not_checked,3.0,NaN
4,lab-1,planner,Креатинин сыворотки крови с расчетом СКФ,,2.3:t04,verified,0.0,NaN
5,lab-2,planner,Общий (клинический) анализ крови,,2.3:t03,verified,0.0,NaN
6,lab-3,planner,Тропонин I (высокочувствительный),,2.3:t01,verified,0.0,NaN
7,lab-4,planner,Тропонин I (высокочувствительный) повторное оп...,,2.3:t02,verified,0.0,NaN
8,lab-5,planner,"Электролиты крови (K, Na)",,2.3:t07,verified,0.0,NaN
9,lab-6,planner,"Липидный профиль (ХС, ЛНП, ЛВП, ТГ)",,2.3:t06,verified,0.0,NaN


ч,0.0,1.0,3.0,24.0,48.0,72.0
показатель,,,,,,
Гемоглобин,118.0,NaN,NaN,118.0,118.0,114.0
Глюкоза,6.1,NaN,NaN,5.9,6.6,6.0
Калий,4.9,NaN,NaN,5.2,5.4,5.1
Креатинин,135.0,NaN,NaN,131.0,149.0,154.0
Тромбоциты,208.0,NaN,NaN,207.0,212.0,203.0
Тропонин (вч),95.0,97.0,112.0,116.0,59.0,28.0
ХС ЛНП,2.9,NaN,NaN,NaN,NaN,NaN
рСКФ (CKD-EPI),35.1,NaN,NaN,36.3,31.1,30.1



АУДИТ:
[CRITICAL] R5_allergy (clinical) ext-1 - «Эналаприл» при аллергии в анамнезе: эналаприл (ингибитор АПФ) (ангионевротический отёк)
    основание: -
[CRITICAL] LLM (clinical) ext-1 - Назначение эналаприла при аллергии на ингибиторы АПФ (ангионевротический отек) является противопоказанным. Согласно КР, при непереносимости иАПФ следует использовать сартаны (валсартан уже запланирован в base-2). Эналаприл нужно отменить немедленно. (факт: Назначен эналаприл (ext-1) при наличии аллергии на ингибиторы АПФ (ангионевротический отек в анамнезе). В треке также запланирован валсартан (base-2) — препарат из группы сартанов, который рекомендован как альтернатива при непереносимости иАПФ.)
    основание: 154_4:3.2.7:t01: «Противопоказания для начала использования иАПФ: систолическое АД < 100 мм рт. ст., выражен…»
    проверка второй моделью: Тезис КР указывает на необходимость применения иАПФ при ФВ ЛЖ < 40% (у пациента 38%) при отсутствии противопоказаний, к которым относится индивидуальная 

## Кейс 3 - Reactive
СД 2, рСКФ 48, КГ+ЧКВ с 260 мл контраста → к 3-м суткам контраст-индуцированное ОПП. Триггеры пересмотра: ОПП, эноксапарин 2 раза/сут от палатного врача при рСКФ < 30, гиперкалиемия, этапное ЧКВ с контрастом. Аудитор проверяет весь трек, включая согласие, оформленное после вмешательства.

In [6]:
show_case(runs['case3'])

## case3: Осложнение: контраст-индуцированное ОПП на 3-и сутки госпитализации

Ангинозный приступ 3 ч назад, купирован нитроглицерином, в приёмном отделении боли нет. СД 2 типа 12 лет (метформин). ЧСС 92 уд/мин, АД 168/94 мм рт. ст., Killip I. ЭКГ: депрессия ST 1 мм в V2-V5, без динамики при повторной регистрации. ЭхоКГ: ФВ ЛЖ 47%.


,id,источник,назначение,доза/режим,ссылка,проверка,"с, ч","до, ч"
0,proc-1,external,Коронарография + ЧКВ со стентированием ПНА,,,not_checked,14.0,NaN
1,proc-2,external,Этапное ЧКВ огибающей и правой коронарной артерий,,,not_checked,96.0,72.0
2,ext-1,external,Эноксапарин натрия,"80 мг (1 мг/кг), п/к, 2 раза в сутки",,not_checked,20.0,72.0
3,lab-1,planner,Креатинин сыворотки крови,,2.3:t04,verified,0.0,NaN
4,lab-2,planner,Общий (клинический) анализ крови,,2.3:t03,verified,0.0,NaN
5,lab-3,planner,Тропонин I (или T) высокочувствительный,,2.3:t01,verified,0.0,NaN
6,lab-4,planner,Тропонин I (или T) высокочувствительный (повто...,,2.3:t02,verified,0.0,NaN
7,lab-5,planner,"Электролиты крови (K, Na)",,2.3:t07,verified,0.0,NaN
8,lab-6,planner,"Липидный профиль (ХС, ЛНП, ЛВП, ТГ)",,2.3:t06,verified,0.0,NaN
9,lab-7,planner,Глюкоза крови,,2.3:t05,verified,0.0,NaN


ч,0.0,1.0,3.0,24.0,48.0,72.0
показатель,,,,,,
Гемоглобин,142.0,NaN,NaN,140.0,137.0,134.0
Глюкоза,11.2,NaN,NaN,11.3,11.3,11.0
Калий,4.6,NaN,NaN,5.1,5.7,5.5
Креатинин,135.0,NaN,NaN,162.0,255.0,302.0
Тромбоциты,251.0,NaN,NaN,258.0,245.0,253.0
Тропонин (вч),180.0,230.0,293.0,434.0,212.0,78.0
ХС ЛНП,3.1,NaN,NaN,NaN,NaN,NaN
рСКФ (CKD-EPI),48.1,NaN,NaN,38.5,22.4,18.3


[critical] Острое повреждение почек, стадия 2: креатинин 135 → 302 мкмоль/л (×2.24), рСКФ 18.3; 260 мл контраста на 14 ч - вероятна контраст-индуцированная нефропатия  | порог: KDIGO 2012 (критерии ОПП; вне КР) + КР 154_4 п. 3.1.3
[warning] рСКФ 18.3: эноксапарин натрия требует коррекции - 1 мг/кг 1 раз в сутки (< 30)  | порог: КР 154_4 п. А3, табл. 4 (строка 11) и табл. 5
[critical] Запланировано «Этапное ЧКВ огибающей и правой коронарной артерий» на 96 ч с 200 мл контраста при рСКФ 18.3: отношение 10.9 ≥ 3,7  | порог: КР 154_4 п. 3.1.3, тезис 3
[warning] Гиперкалиемия 5.7 ммоль/л  | порог: КР 154_4 п. 2.3, тезис 7 (контроль электролитов)

ПЕРЕСМОТР на 72.0 ч: У пациента развилась контраст-индуцированная нефропатия (рСКФ 18.3 мл/мин/1,73 м²) и гиперкалиемия. Эноксапарин требует коррекции дозы или замены на НФГ при рСКФ < 30. Повторное ЧКВ откладывается из-за высокого риска дальнейшего повреждения почек (соотношение контраст/рСКФ > 3.7). Метформин отменен из-за ОПП.
  - отменено ext-1 

## Сводка

In [7]:
pd.DataFrame([{'кейс': cid, 'назначений в плане': sum(i.source == 'planner' for i in r.plan_initial.items),
  'с подтверждённой ссылкой': sum(i.verification == 'verified' for i in r.plan_initial.items if i.source == 'planner'),
  'отклонений': len(r.deviations), 'находок аудитора': len(r.audit.findings), **r.audit.by_severity(),
  'отклонено проверкой': len(r.audit.rejected),
  'recall заложенных дефектов': r.coverage['recall']} for cid, r in runs.items()])

,кейс,назначений в плане,с подтверждённой ссылкой,отклонений,находок аудитора,critical,major,minor,отклонено проверкой,recall заложенных дефектов
0,case1,16,16,0,0,0,0,0,3,NaN
1,case2,18,18,0,5,2,2,1,1,1.0
2,case3,17,17,4,5,1,4,0,2,1.0
